# Notebook 3: การใช้ LLM ช่วยจัดโครงสร้างข้อมูล
# (LLM-Assisted Data Structuring with Ollama)

## สิ่งที่จะได้เรียนรู้
- ตั้งค่า Ollama กับ Qwen2.5
- ออกแบบ prompt สำหรับสร้างคู่คำถาม-คำตอบภาษาไทย
- สร้าง Q&A pairs จาก chunks
- กรองคุณภาพข้อมูล (quality filtering)
- บันทึกเป็น Alpaca format สำหรับ fine-tuning

In [ ]:
# ติดตั้ง dependencies (Install dependencies)
%pip install ollama pythainlp tqdm

In [ ]:
import ollama

# ทดสอบเชื่อมต่อ Ollama
try:
    models = ollama.list()
    print("เชื่อมต่อ Ollama สำเร็จ!")
    print("โมเดลที่มี:")
    for model in models.models:
        print(f"  - {model.model}")
except Exception as e:
    print(f"เชื่อมต่อ Ollama ไม่สำเร็จ: {e}")
    print("กรุณาตรวจสอบว่า Ollama กำลังทำงาน (ollama serve)")

In [ ]:
# เลือกโมเดลที่จะใช้ (เปลี่ยนได้ตามที่มีใน Ollama)
MODEL_NAME = "gemma3:27b-cloud"  # เปลี่ยนเป็นโมเดลที่มีอยู่ เช่น "qwen2.5:3b" ถ้าติดตั้งไว้

# ทดสอบสร้างข้อความภาษาไทย
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "สวัสดีครับ ช่วยอธิบายว่า AI คืออะไรสั้นๆ เป็นภาษาไทย"}
    ]
)

print(f"ใช้โมเดล: {MODEL_NAME}")
print("คำตอบจากโมเดล:")
print(response["message"]["content"])

In [ ]:
QA_PROMPT_TEMPLATE = """คุณเป็นผู้เชี่ยวชาญในการสร้างชุดข้อมูลสำหรับสอนโมเดล AI
จากข้อความด้านล่าง กรุณาสร้างคู่คำถาม-คำตอบ 3 คู่ เป็นภาษาไทย

กฎ:
- คำถามต้องตอบได้จากข้อความที่ให้มาเท่านั้น
- คำตอบต้องถูกต้องและครบถ้วน
- ใช้ภาษาไทยทั้งหมด
- ตอบในรูปแบบ JSON ที่กำหนด

ข้อความ:
{chunk_text}

ตอบเป็น JSON array ดังนี้:
[
  {{"instruction": "คำถามที่ 1", "input": "", "output": "คำตอบที่ 1"}},
  {{"instruction": "คำถามที่ 2", "input": "", "output": "คำตอบที่ 2"}},
  {{"instruction": "คำถามที่ 3", "input": "", "output": "คำตอบที่ 3"}}
]

JSON:"""

# ทดสอบกับข้อความตัวอย่าง
test_chunk = "ปัญญาประดิษฐ์ หรือ AI คือศาสตร์ที่เกี่ยวข้องกับการสร้างเครื่องจักรที่สามารถทำงานที่ต้องใช้สติปัญญาของมนุษย์ เช่น การเรียนรู้ การตัดสินใจ และการแก้ปัญหา"

prompt = QA_PROMPT_TEMPLATE.format(chunk_text=test_chunk)
print("Prompt ที่สร้าง:")
print(prompt[:500])

In [ ]:
import json
import re

def generate_qa_pairs(chunk_text: str, model: str = MODEL_NAME) -> list[dict]:
    """สร้างคู่คำถาม-คำตอบจาก chunk ด้วย Ollama"""
    prompt = QA_PROMPT_TEMPLATE.format(chunk_text=chunk_text)

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3}
    )

    raw_output = response["message"]["content"]

    # พยายาม parse JSON จากคำตอบ
    try:
        json_match = re.search(r'\[.*\]', raw_output, re.DOTALL)
        if json_match:
            qa_pairs = json.loads(json_match.group())
            return qa_pairs
    except json.JSONDecodeError:
        pass

    return []

# ทดสอบ
test_pairs = generate_qa_pairs(test_chunk)
print(f"สร้างได้ {len(test_pairs)} คู่:")
print(json.dumps(test_pairs, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
from tqdm import tqdm
import time

# โหลด chunks จาก Notebook 2
chunks_path = Path("../output/chunks/chunks.jsonl")
chunks = []
with open(chunks_path, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"โหลด {len(chunks)} chunks")

# สร้าง Q&A pairs สำหรับแต่ละ chunk
all_qa_pairs = []
failed_chunks = 0

for chunk in tqdm(chunks, desc="กำลังสร้าง Q&A"):
    try:
        pairs = generate_qa_pairs(chunk["text"])

        for pair in pairs:
            pair["source_chunk_id"] = chunk["chunk_id"]
            pair["source_file"] = chunk["metadata"]["source"]

        all_qa_pairs.extend(pairs)
    except Exception as e:
        failed_chunks += 1
        continue

    time.sleep(0.1)

print(f"\nสรุป:")
print(f"  สร้าง Q&A pairs: {len(all_qa_pairs)}")
print(f"  Chunks ที่ล้มเหลว: {failed_chunks}")

In [ ]:
from pythainlp.util import isthai

def quality_filter(qa_pairs: list[dict]) -> list[dict]:
    """กรองคุณภาพ Q&A pairs"""
    filtered = []
    reasons = {"too_short": 0, "no_thai": 0, "duplicate": 0, "passed": 0}
    seen_instructions = set()

    for pair in qa_pairs:
        instruction = pair.get("instruction", "")
        output = pair.get("output", "")

        # 1. ตรวจความยาวขั้นต่ำ
        if len(instruction) < 10 or len(output) < 10:
            reasons["too_short"] += 1
            continue

        # 2. ตรวจว่ามีภาษาไทย
        thai_chars = sum(1 for c in instruction + output if '\u0e00' <= c <= '\u0e7f')
        total_chars = len(instruction + output)
        if total_chars > 0 and thai_chars / total_chars < 0.3:
            reasons["no_thai"] += 1
            continue

        # 3. ตรวจซ้ำ
        if instruction in seen_instructions:
            reasons["duplicate"] += 1
            continue
        seen_instructions.add(instruction)

        reasons["passed"] += 1
        filtered.append(pair)

    print("ผลการกรองคุณภาพ:")
    for reason, count in reasons.items():
        print(f"  {reason}: {count}")

    return filtered

# กรองคุณภาพ
filtered_pairs = quality_filter(all_qa_pairs)
print(f"\nเหลือ {len(filtered_pairs)} Q&A pairs หลังกรอง")

In [ ]:
# บันทึกเป็น Alpaca format
output_dir = Path("../output/datasets")
output_dir.mkdir(parents=True, exist_ok=True)

sft_path = output_dir / "sft_dataset.jsonl"

with open(sft_path, "w", encoding="utf-8") as f:
    for pair in filtered_pairs:
        alpaca_entry = {
            "instruction": pair["instruction"],
            "input": pair.get("input", ""),
            "output": pair["output"],
        }
        f.write(json.dumps(alpaca_entry, ensure_ascii=False) + "\n")

print(f"บันทึก SFT dataset: {sft_path}")
print(f"จำนวน: {len(filtered_pairs)} entries")

# แสดงตัวอย่าง
print("\nตัวอย่าง 2 entries แรก:")
for pair in filtered_pairs[:2]:
    print(json.dumps(
        {"instruction": pair["instruction"], "input": pair.get("input", ""), "output": pair["output"]},
        ensure_ascii=False, indent=2
    ))
    print()